### Establish connection

In [5]:
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv
from datetime import datetime, timezone
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

try:
    from dateutil import parser as date_parser
except ImportError:
    date_parser = None

load_dotenv()

True

In [6]:
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
ARTICLES_COLLECTION_NAME = os.getenv("MONGO_COLLECTION_NAME")

if not all([MONGO_URI, DB_NAME, ARTICLES_COLLECTION_NAME]):
    raise ValueError("❌ Missing required environment variables. Check your .env file.")

try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]
    articles_collection = db[ARTICLES_COLLECTION_NAME]
    client.admin.command("ping")
    print(f"✅ Connected to MongoDB Atlas! Database: {DB_NAME}")
except Exception as e:
    raise RuntimeError(f"❌ MongoDB Connection Error: {e}")

✅ Connected to MongoDB Atlas! Database: tuone


In [7]:
# Return unique instances of a variable

unique_validations = articles_collection.distinct("llm_processed")
print(unique_validations)

# return count of entries with a given variable 

count_with_validation = articles_collection.count_documents({"llm_processed": {"$exists": True}})
print(count_with_validation) 

[{'run_id': 'v0', 'ts': '2025-05-15T16:08:23.263525+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:04:26.530588+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:04:49.631834+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:04:55.617248+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:05:04.196473+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:05:15.551463+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:05:31.935233+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:05:41.073583+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:05:59.586005+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:06:07.082696+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:06:12.792778+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:06:19.346487+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:06:31.736662+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:06:46.994597+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:07:01.556099+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:07:12.704437+00:00'}, {'run_id': 'v0', 'ts': '2025-05-15T20:0

In [8]:
results = list(articles_collection.find(
    {"validated": {"$exists": True}},
    {"title": 1, "meta.date": 1, "_id": 0}
))

titles = [doc["title"] for doc in results]
print(titles)

dates = [doc["meta"]["date"] for doc in results if "meta" in doc and "date" in doc["meta"]]
print(dates)

['Mahle integrates DC charger to climatic wind tunnel', 'BMW launches operation of battery test centre in Wackersdorf', 'Cornish Lithium opens demonstration plant in Cornwall', 'Funding for battery recycling plant from ABTC', 'Moment Energy to construct EV battery reuse Gigafactory in Texas', 'Forsee Power inaugurates factory in Ohio', 'Ford pauses construction of Canadian cathode material plant again', 'BTR plans to produce anode materials in Morocco', 'Funding for battery recycling plant from ABTC', 'Elgin Energy Finds Greek Partner for 76MW UK PV', 'Sterling and Wilson Wins $141 Million Deal for 500 MW Solar Project in India']
['2023-02-09T00:00:00.000Z', '2024-11-19T00:00:00.000Z', '2024-10-18T00:00:00.000Z', '2024-12-20T00:00:00.000Z', '2024-10-29T00:00:00.000Z', '2024-09-13T00:00:00.000Z', '2024-08-21T00:00:00.000Z', '2024-08-26T00:00:00.000Z', '2024-12-20T00:00:00.000Z', '2020-07-09T00:00:00.000Z', '2024-12-24T00:00:00.000Z']


In [9]:
results

[{'title': 'Mahle integrates DC charger to climatic wind tunnel',
  'meta': {'date': '2023-02-09T00:00:00.000Z'}},
 {'title': 'BMW launches operation of battery test centre in Wackersdorf',
  'meta': {'date': '2024-11-19T00:00:00.000Z'}},
 {'title': 'Cornish Lithium opens demonstration plant in Cornwall',
  'meta': {'date': '2024-10-18T00:00:00.000Z'}},
 {'title': 'Funding for battery recycling plant from ABTC',
  'meta': {'date': '2024-12-20T00:00:00.000Z'}},
 {'title': 'Moment Energy to construct EV battery reuse Gigafactory in Texas',
  'meta': {'date': '2024-10-29T00:00:00.000Z'}},
 {'title': 'Forsee Power inaugurates factory in Ohio',
  'meta': {'date': '2024-09-13T00:00:00.000Z'}},
 {'title': 'Ford pauses construction of Canadian cathode material plant again',
  'meta': {'date': '2024-08-21T00:00:00.000Z'}},
 {'title': 'BTR plans to produce anode materials in Morocco',
  'meta': {'date': '2024-08-26T00:00:00.000Z'}},
 {'title': 'Funding for battery recycling plant from ABTC',
  '

In [34]:

pipeline = [
    {"$project": {"fieldNames": {"$objectToArray": "$$ROOT"}}},
    {"$unwind": "$fieldNames"},
    {"$group": {"_id": None, "allFields": {"$addToSet": "$fieldNames.k"}}}
]

result = list(articles_collection.aggregate(pipeline))
field_names = result[0]["allFields"] if result else []

In [35]:
field_names

['nodes',
 'relationships',
 'title',
 'llm_processed',
 '_id',
 'comment',
 'meta',
 'paragraphs',
 'validation',
 'validated']

### General search

In [10]:
offset = 0
limit = 1

articles_to_process = list(
    articles_collection.find(
        {"meta.ID": {"$regex": "^13"}}, 
        {"_id": 1, "paragraphs": 1, "validation": 1, "meta": 1, "title": 1,
         "nodes":1, "relationships":1}
    )
    .sort("_id", -1)  # Sort by MongoDB ObjectId (descending)
    .skip(offset)    # Skip first `offset` articles
    .limit(limit)    # Limit the number of articles
)

for article in articles_to_process:
    print(article["title"])

    val = article.get("validation")

    if val is True:
        print("⏭️  Skipping – article is validated")
        continue
    elif isinstance(val, (int, float)):           # timestamp
        processed_on = datetime.fromtimestamp(val, tz=timezone.utc) \
                                 .strftime("%Y‑%m‑%d %H:%M UTC")
        print(f"⏭️  Skipping – article was validated on {processed_on}")
        continue

Ten Most Read News on OffshoreWIND.biz in 2024
⏭️  Skipping – article was validated on 1970‑01‑01 00:00 UTC


In [11]:
articles_to_process

[{'_id': ObjectId('67fa5f2e38fb90ab92f8ba73'),
  'title': 'Ten Most Read News on OffshoreWIND.biz in 2024',
  'paragraphs': [{'p1': 'Share this article',
    'p2': 'New wind turbine models, vessels and equipment, and project updates were the most read topics onoffshoreWIND.bizin 2024. In this overview, we are bringing the ten most popular news in 2024.',
    'p3': 'In January 2024, Siemens Gamesa received a EUR 30 million grant from the EU for a project calledHighly Innovative Prototype of the most Powerful Offshore Wind turbine generator(HIPPOW).',
    'p4': 'According to the EU document on the grant award, the project involves the installation, operation, and testing of “the world’s most powerful wind turbine prototype” at the Østerild National\xa0Test Centre\xa0in Denmark to validate new technological developments and obtain necessary certifications before starting full-scale production.',
    'p5': 'At the end of the year, it was reported that Siemens Gamesa wastransporting a wind 

### Validation metrics

In [4]:
validated_count = articles_collection.count_documents({"validation": True})
print(f"Number of validated articles: {validated_count}")

Number of validated articles: 168


### Publication date metrics

In [5]:
# Check meta.date consistency
def classify_date(value):
    if isinstance(value, datetime):
        return "bson"
    if isinstance(value, str):
        if date_parser:
            try:
                date_parser.parse(value)
                return "string"
            except (ValueError, OverflowError):
                return "invalid"
    return "invalid"

In [6]:
counts = {"bson": 0, "string": 0, "invalid": 0, "missing": 0}
invalid_ids = []

cursor = articles_collection.find({}, {"_id": 1, "meta.date": 1}, batch_size=10_000)
total = 0

for doc in cursor:
    total += 1
    meta = doc.get("meta", {})
    if "date" not in meta:
        counts["missing"] += 1
        invalid_ids.append(doc["_id"])
        continue
    category = classify_date(meta["date"])
    counts[category] += 1
    if category == "invalid":
        invalid_ids.append(doc["_id"])

In [7]:
print("\n--- meta.date consistency report ---")
print(f"Total documents: {total:,}")
for key in ["bson", "string", "invalid", "missing"]:
    n = counts[key]
    pct = (n / total * 100) if total else 0
    print(f"{key.capitalize():>8}: {n:>7,}  ({pct:5.1f}%)")


--- meta.date consistency report ---
Total documents: 39,578
    Bson:  39,318  ( 99.3%)
  String:      57  (  0.1%)
 Invalid:     203  (  0.5%)
 Missing:       0  (  0.0%)


In [8]:
# return all date entries
cursor = articles_collection.find(
    {"meta.date": {"$type": "date"}},
    {"_id": 0, "meta.date": 1}
)
dates = [doc["meta"]["date"].date() for doc in tqdm(cursor)]
df = pd.DataFrame(dates, columns=["date"])

daily_counts = df.value_counts().reset_index(name="count").sort_values("date")
daily_counts.columns = ["date", "count"]
daily_counts["date"] = pd.to_datetime(daily_counts["date"])  # ensure correct dtype
daily_counts["year"] = daily_counts["date"].dt.year

output_dir = "descriptive/daily_counts"

# save one plot per year
years = sorted(daily_counts["year"].unique())
for year in years:
    fig, ax = plt.subplots(figsize=(12, 4))
    year_df = daily_counts[daily_counts["year"] == year]
    ax.plot(year_df["date"], year_df["count"], marker="", linewidth=1)
    ax.set_title(f"Daily Article Count – {year}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Articles per Day")
    ax.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()

    fig_path = os.path.join(output_dir, f"daily_count_{year}.png")
    plt.savefig(fig_path)
    plt.close(fig)

print(f"✅ Saved {len(years)} plots to {output_dir}")

39318it [00:00, 109476.04it/s]


✅ Saved 19 plots to descriptive/daily_counts


### Duplicate entries

In [9]:
# return all URLs
cursor = articles_collection.find(
    {"meta.url": {"$exists": True}},  # ensure url field exists
    {"_id": 1, "meta.url": 1}
)

# build dataframe
data = [{"_id": doc["_id"], "url": doc["meta"]["url"]} for doc in tqdm(cursor)]
df_urls = pd.DataFrame(data)

# count duplicate URLs
url_counts = df_urls["url"].value_counts()
duplicates = url_counts[url_counts > 1]

# display result
print(f"🔍 Total articles: {len(df_urls)}")
print(f"❗ Duplicate URLs found: {len(duplicates)}")
print(duplicates.head(40))  # top 10 most duplicated URLs

38564it [00:01, 37105.42it/s]

🔍 Total articles: 38564
❗ Duplicate URLs found: 1401
url
https://www.world-energy.org/tag/150.html                                                                                                8
https://www.world-energy.org/tag/244.html                                                                                                7
https://www.world-energy.org/tag/176.html                                                                                                6
https://www.pv-magazine.com/author/sergiomatalucci/                                                                                      5
https://www.pv-magazine.com/author/uma-gupta/                                                                                            5
https://www.world-energy.org/article/39139.html                                                                                          5
https://www.world-energy.org/article/32160.html                                                              

In [10]:
# Step 1: Query for all titles
cursor = articles_collection.find(
    {"title": {"$exists": True}},  # only documents with title
    {"_id": 1, "title": 1}
)

# Step 2: Build dataframe
data = [{"_id": doc["_id"], "title": doc["title"]} for doc in tqdm(cursor)]
df_titles = pd.DataFrame(data)

# Step 3: Count duplicate titles
title_counts = df_titles["title"].value_counts()
duplicate_titles = title_counts[title_counts > 1]

# Step 4: Display result
print(f"🔍 Total articles: {len(df_titles)}")
print(f"❗ Duplicate titles found: {len(duplicate_titles)}")
print(duplicate_titles.head(40))

38564it [00:00, 52900.53it/s]

🔍 Total articles: 38564
❗ Duplicate titles found: 1656
title
No Title Found                                                                                        1590
Utility Scale PV                                                                                       601
Modules & Upstream Manufacturing                                                                       412
Hydrogen                                                                                                72
                                                                                                        48
Energy Storage                                                                                          35
Balance of Systems                                                                                      26
The pv magazine weekly news digest                                                                      11
Solar Power Plant                                                                  

In [17]:
# Step 1: Filter only titles that are duplicates
duplicate_title_counts = title_counts[title_counts > 1]

# Step 2: Count how many titles appear 2 times, 3 times, etc.
dup_frequency_distribution = duplicate_title_counts.value_counts().sort_index(ascending=False)

# Step 3: Display the distribution
print("🧮 Distribution of duplicate title frequencies:")
print(dup_frequency_distribution)

🧮 Distribution of duplicate title frequencies:
count
1592       1
601        1
412        1
73         1
35         1
26         1
11         1
10         1
8          3
7          3
6          9
5         15
4         80
3        257
2       1302
Name: count, dtype: int64


### by article source category

In [9]:
pipeline = [
    {"$group": {"_id": "$meta.category", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}         # optional, highest count first
]

for doc in articles_collection.aggregate(pipeline):
    print(f"{doc['_id']}: {doc['count']}")

world_energy: 9808
pvmagazine: 9433
pvtech: 7709
electrive: 4868
renewsBiz: 4360
powertechnology: 1437
offshorewind: 665
energy_voice_hydrogen: 284


In [14]:
from pprint import pprint 
pipeline = [
    # 1. Collapse to the two fields we care about
    {
        "$project": {
            "category": {"$ifNull": ["$meta.category", "⟨no-category⟩"]},
            "run_id":   "$llm_processed.run_id"
        }
    },

    # 2. Group by BOTH category and run_id
    {
        "$group": {
            "_id": {"category": "$category", "run_id": "$run_id"},
            "count": {"$sum": 1}
        }
    },

    # 3. Regroup by category to bundle the run_id breakdown
    {
        "$group": {
            "_id": "$_id.category",
            "total":   {"$sum": "$count"},
            "by_run":  {
                "$push": {         # keep each run-id + its count
                    "run_id": "$_id.run_id",
                    "count" : "$count"
                }
            }
        }
    },

    { "$sort": { "total": -1 } }               # optional pretty ordering
]

results = list(articles_collection.aggregate(pipeline))
pprint(results)

[{'_id': 'world_energy',
  'by_run': [{'count': 8813}, {'count': 995, 'run_id': 'v0'}],
  'total': 9808},
 {'_id': 'pvmagazine', 'by_run': [{'count': 9433}], 'total': 9433},
 {'_id': 'pvtech', 'by_run': [{'count': 7709}], 'total': 7709},
 {'_id': 'electrive',
  'by_run': [{'count': 445, 'run_id': 'v0'}, {'count': 4423}],
  'total': 4868},
 {'_id': 'renewsBiz', 'by_run': [{'count': 4360}], 'total': 4360},
 {'_id': 'powertechnology', 'by_run': [{'count': 1437}], 'total': 1437},
 {'_id': 'offshorewind', 'by_run': [{'count': 665}], 'total': 665},
 {'_id': 'energy_voice_hydrogen', 'by_run': [{'count': 284}], 'total': 284}]
